**Legacy notebook.** Self-contained analysis code that predates the `src/mrvf` library and has not been ported to it. Kept for provenance and because it still produces figures in `results/`. Paths were updated to the `results/` layout; the next cell sets the working directory to the repository root, so run it from anywhere.

For the maintained pipeline see `notebooks/01_train_triple_regime.ipynb` and `notebooks/02_evaluate_rmse_vs_snr.ipynb`.

In [ ]:
import os
from pathlib import Path
# run from the repository root so ./results/... and ../subsamples resolve
_root = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "src" / "mrvf").is_dir())
os.chdir(_root)

# Paired voxel cross-section + signal curve

For each vascular parameter combination, shows:
- **Left**: schematic 2D voxel cross-section (vessel count encodes CBV, circle size encodes R)
- **Right**: median normalised signal curve from matched dictionary entries, with 10–90th percentile band

Intended for slide 11 — illustrates that 1,599,000 unique parameter combinations each produce a unique signal fingerprint.

**Requires** the data-loading cells from `visualize_dictionary_signals.ipynb` to have been run first,
or paste those cells here (Sections 1–3 of that notebook).

## 1  Imports & paths
Skip this cell if you are running after `visualize_dictionary_signals.ipynb`.

In [ ]:
import numpy as np
import scipy.io as sio
import h5py
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Circle

plt.rcParams.update({
    'font.family'    : 'Arial',
    'font.size'      : 9,
    'axes.labelsize' : 9,
    'axes.titlesize' : 8.5,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'figure.dpi'     : 150,
    'savefig.dpi'    : 300,
    'savefig.bbox'   : 'tight',
})

# ── Paths — adjust to your directory layout ─────────────────────────────
SIGNAL_PATH   = '../subsamples/subsamples_v3/QuasiRand_t2_200.mat'
PARAM_PATH    = '../subsamples/subsamples_v3/QuasiRand_par_t2_200.mat'
ECHOTIME_PATH = '../echotimes.mat'

SIGNAL_KEY    = 'Dico40_save'
PARAM_KEY     = 'par_save'
ECHOTIME_KEY  = 'Echotimes'

print('Imports OK')

## 2  Load data
Skip this cell if variables are already in memory.

In [ ]:
def load_mat(path, key):
    """Load .mat — handles v5 (scipy) and v7.3 (h5py) formats."""
    try:
        mat = sio.loadmat(path)
        if key in mat:
            return np.array(mat[key], dtype=np.float32)
        cands = [k for k in mat if not k.startswith('_')]
        print(f'  Key "{key}" not found; using "{cands[0]}"')
        return np.array(mat[cands[0]], dtype=np.float32)
    except NotImplementedError:
        with h5py.File(path, 'r') as f:
            if key in f:
                data = f[key][()]
            else:
                first = next(k for k in f if not k.startswith('#'))
                print(f'  Key "{key}" not found; using "{first}"')
                data = f[first][()]
            if data.ndim >= 2:
                data = data.T
            return np.array(data, dtype=np.float32)


signals = load_mat(SIGNAL_PATH, SIGNAL_KEY)
params  = load_mat(PARAM_PATH,  PARAM_KEY)[:, :4]

et_raw     = load_mat(ECHOTIME_PATH, ECHOTIME_KEY).flatten()
echo_times = et_raw if et_raw.max() > 1 else et_raw * 1000   # ensure ms

SO2 = params[:, 0] * 100    # fraction → %
CBV = params[:, 1] * 100    # fraction → %
R   = params[:, 2] * 1e6    # m → µm
T2  = params[:, 3] * 1000   # s → ms

print(f'Signals : {signals.shape}')
print(f'Params  : {params.shape}   (SO2, CBV, R, T2)')
print(f'Echoes  : {echo_times.shape}  [{echo_times[0]:.1f} … {echo_times[-1]:.1f}] ms')

## 3  Query helper

In [ ]:
def find_signals(target_so2=None, target_cbv=None, target_r=None, target_t2=None,
                 n=20, tol_so2=5, tol_cbv=0.8, tol_r=2.5, tol_t2=15, seed=0):
    """
    Return indices of signals whose parameters are within tolerance of targets.
    Parameters set to None are unconstrained.
    """
    mask = np.ones(len(signals), dtype=bool)
    if target_so2 is not None: mask &= np.abs(SO2 - target_so2) <= tol_so2
    if target_cbv is not None: mask &= np.abs(CBV - target_cbv) <= tol_cbv
    if target_r   is not None: mask &= np.abs(R   - target_r)   <= tol_r
    if target_t2  is not None: mask &= np.abs(T2  - target_t2)  <= tol_t2

    candidates = np.where(mask)[0]
    if len(candidates) == 0:
        raise ValueError(
            f'No signals found. Try relaxing tolerances. '
            f'(tol_so2={tol_so2}, tol_cbv={tol_cbv}, tol_r={tol_r}, tol_t2={tol_t2})'
        )
    rng = np.random.default_rng(seed)
    return np.sort(rng.choice(candidates, size=min(n, len(candidates)), replace=False))

print('Query helper ready.')

## 4  Define cases
Edit this cell to change which parameter combinations are shown.

In [ ]:
# Each entry defines one row in the figure.
# vessel_r_frac : radius of drawn circles as fraction of the voxel half-width (0–0.5)
# n_vessels     : approximate number of vessels to draw in the schematic

cases = [
    dict(
        label        = 'Low CBV, small vessels\n(CBV = 2%,  R = 5 µm)',
        color        = '#1565C0',
        n_vessels    = 7,
        vessel_r_frac= 0.035,
        query        = dict(target_so2=70, target_cbv=2,  target_r=5,  target_t2=80,
                            n=30, tol_so2=5, tol_cbv=0.6, tol_r=2, tol_t2=15),
    ),
    dict(
        label        = 'High CBV, small vessels\n(CBV = 12%,  R = 5 µm)',
        color        = '#C62828',
        n_vessels    = 24,
        vessel_r_frac= 0.035,
        query        = dict(target_so2=70, target_cbv=12, target_r=5,  target_t2=80,
                            n=30, tol_so2=5, tol_cbv=1.0, tol_r=2, tol_t2=15),
    ),
    dict(
        label        = 'Low CBV, large vessels\n(CBV = 4%,  R = 20 µm)',
        color        = '#6A1B9A',
        n_vessels    = 4,
        vessel_r_frac= 0.14,
        query        = dict(target_so2=70, target_cbv=4,  target_r=20, target_t2=80,
                            n=30, tol_so2=5, tol_cbv=0.8, tol_r=2.5, tol_t2=15),
    ),
    dict(
        label        = 'High CBV, large vessels\n(CBV = 12%,  R = 20 µm)',
        color        = '#BF360C',
        n_vessels    = 14,
        vessel_r_frac= 0.14,
        query        = dict(target_so2=70, target_cbv=12, target_r=20, target_t2=80,
                            n=30, tol_so2=5, tol_cbv=1.0, tol_r=2.5, tol_t2=15),
    ),
]

print(f'{len(cases)} cases defined.')

## 5  Draw figure

In [ ]:
n_rows = len(cases)
fig    = plt.figure(figsize=(7.5, 2.1 * n_rows))
gs     = gridspec.GridSpec(
    n_rows, 2,
    width_ratios=[1, 2.8],
    hspace=0.6, wspace=0.18,
    left=0.05, right=0.97, top=0.93, bottom=0.06,
)

rng    = np.random.default_rng(42)
se_idx = 29                            # spin echo at echo index 29

for row, case in enumerate(cases):

    # ── left panel: schematic voxel cross-section ────────────────────────
    ax_vox = fig.add_subplot(gs[row, 0])
    ax_vox.set_xlim(0, 1)
    ax_vox.set_ylim(0, 1)
    ax_vox.set_aspect('equal')
    ax_vox.set_facecolor('#F6F4EF')
    for sp in ax_vox.spines.values():
        sp.set_linewidth(0.5)
        sp.set_color('#AAAAAA')
    ax_vox.set_xticks([])
    ax_vox.set_yticks([])

    r      = case['vessel_r_frac']
    margin = r + 0.04
    placed = []
    attempts = 0
    while len(placed) < case['n_vessels'] and attempts < 8000:
        cx = rng.uniform(margin, 1 - margin)
        cy = rng.uniform(margin, 1 - margin)
        if all(np.hypot(cx - px, cy - py) > 2.3 * r for px, py in placed):
            placed.append((cx, cy))
            ax_vox.add_patch(Circle(
                (cx, cy), r,
                facecolor='#7B1515', edgecolor='#350505',
                linewidth=0.6, zorder=3,
            ))
        attempts += 1

    ax_vox.set_title(case['label'], fontsize=7.5, loc='center', pad=3, color='#333333')

    # ── right panel: median signal + IQR band ────────────────────────────
    ax_sig = fig.add_subplot(gs[row, 1])

    idx  = find_signals(**case['query'])
    sigs = signals[idx].astype(np.float64)
    nrm  = np.linalg.norm(sigs, axis=1, keepdims=True)
    sigs = sigs / np.maximum(nrm, 1e-12)

    med = np.median(sigs, axis=0)
    lo  = np.percentile(sigs, 10, axis=0)
    hi  = np.percentile(sigs, 90, axis=0)

    ax_sig.fill_between(echo_times, lo, hi,
                        color=case['color'], alpha=0.12, linewidth=0)
    ax_sig.plot(echo_times, med,
                color=case['color'], linewidth=2,
                solid_capstyle='round', zorder=3)

    # subtle vertical line at spin echo
    # ax_sig.axvline(echo_times[se_idx], color='#CCCCCC',
    #                linewidth=0.7, linestyle=':', zorder=1)

    # Part A / B / C labels on first row only
    # if row == 0:
    #     ymax = ax_sig.get_ylim()[1]
    #     for xm, lbl in [
    #         (echo_times[6],  'A'),
    #         (echo_times[21], 'B'),
    #         (echo_times[35], 'C'),
    #     ]:
    #         ax_sig.text(xm, ymax * 0.96, lbl,
    #                     ha='center', va='top',
    #                     fontsize=7.5, color='#AAAAAA', style='italic')

    ax_sig.set_xlim(echo_times[0], echo_times[-1])
    ax_sig.set_ylim(bottom=0)
    ax_sig.set_xlabel('Echo time (ms)', fontsize=8)
    if row == 0:
        ax_sig.set_ylabel('Normalised signal', fontsize=8)
    ax_sig.tick_params(labelsize=7.5)
    ax_sig.spines[['top', 'right']].set_visible(False)
    ax_sig.spines[['left', 'bottom']].set_linewidth(0.5)

    print(f'Row {row+1}: {len(idx)} matched signals  |  {case["label"].split(chr(10))[0]}')

fig.suptitle(
    'Different vascular geometries → different signal fingerprints   '
    '(SO₂ = 70%, T2 = 80 ms fixed)',
    fontsize=9, y=0.99,
)

plt.savefig('results/figures/paired_voxel_signal.png', dpi=300)
plt.savefig('results/figures/paired_voxel_signal.pdf')
plt.show()
print('\nSaved results/figures/paired_voxel_signal.png / .pdf')

## 6  Tuning tips

**If `find_signals` raises a `ValueError`** (no matching entries found):
- Relax `tol_cbv` or `tol_r` in the query dict for that case.
- Check that the target value is within the dictionary's parameter range using the printout from Section 2.

**To change which cases are shown**: edit the `cases` list in Section 4.
- `n_vessels` and `vessel_r_frac` only affect the schematic drawing — they do not need to be physically exact.
  Use them to make the visual contrast between rows clearly readable.
- `vessel_r_frac ≈ R_µm / 150` is a rough guide for keeping circles legible in the voxel panel.

**To add more rows**: append entries to `cases` and re-run Section 5.

**To export for PowerPoint**: use the `.png` output at 300 dpi, or the `.pdf` for vector editing in Illustrator.